## Apresentação 

Implementação de um modelo que utiliza um contextualizador para a manutenção da dependência de informação durante a interação com o usuário em um contexto de conversational RAG.

### Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import getpass
import logging
import os
from typing import Dict, List

from features.clean_memory import CleanMemory
from IPython.display import Markdown
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain.schema import Document
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.embeddings import Embeddings
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.vectorstores import InMemoryVectorStore, VectorStore
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from prompt.check_context import check_context_prompt
from prompt.contextualize_message import contextualize_prompt
from prompt.simple_system_message import simple_system_prompt
from prompt.system_message import system_prompt

### Inicializando a LLM

In [3]:
# API reference : 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

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [4]:
llama = "llama3-70b-8192"
deepseek = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama, 
    temperature = 0
)

llm.invoke("Não responda sobre o seu processo de reflexão interna, mas apenas a mensagem do usuário: Olá, tudo bem ?").content

'Olá! Sim, tudo bem. E você?'

### Embedding

In [5]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)


CPU times: total: 8.92 s
Wall time: 18.2 s


In [6]:
# Testando a vetorização pelo embedding informado 
# em relação ao texto enviado. 

vector = embeddings.embed_query(text="Whe love is the lay a YOUTopia")

vector[:5]

[0.03670826926827431,
 0.01380241010338068,
 -0.0006082257023081183,
 -0.01629478670656681,
 -0.002197320805862546]

### Formando a base de conhecimento

Base de conhecimento, também conhecida como knowledge base se refere a uma fonte de informação a partir da qual o modelo utiliza para responder o usuário, visando garantir um incremento da qualidade de resposta, proporcionando uma não dependência do pré-treinamento dos modelos de LLM. 

In [7]:
%%time

"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

loader = PyPDFLoader("../data/Review of AI and Mental Health.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
).split_documents(loader)    

retriever = InMemoryVectorStore.from_documents( 
    documents = text_splitter,
    embedding = embeddings
).as_retriever(search_kwargs={"k": 3})

CPU times: total: 2min 9s
Wall time: 44.6 s


### Chatbot

In [8]:
class Bimo:
    def __init__(
        self,
        llm: BaseChatModel,
        system_message: PromptTemplate,
        check_context_prompt: PromptTemplate,
        contextualizer_prompt: PromptTemplate,
        memory: BaseChatMessageHistory,
        retriever: BaseRetriever,
        include_memory: bool,
        max_messages: int
    ) -> None:
        """
        Initialize the Bimo conversational agent with its components and settings.

        Args:
            llm (BaseChatModel): Language model for generating responses.
            system_message (PromptTemplate): Template for system-level instructions.
            check_context_prompt (PromptTemplate): Template to check if context is needed.
            contextualizer_prompt (PromptTemplate): Template to rephrase and add context.
            memory (BaseChatMessageHistory): History of past messages.
            retriever (BaseRetriever): Retriever to fetch relevant documents.
            include_memory (bool): Flag to include history in processing.
            max_messages (int): Maximum messages to keep in memory.
        """
        self.llm                   = llm
        self.system_message        = system_message
        self.check_context_prompt  = check_context_prompt
        self.contextualizer_prompt = contextualizer_prompt
        self.memory                = memory
        self.retriever             = retriever
        self.include_memory        = include_memory
        self.max_messages          = max_messages
        self.logger                = logging.getLogger(__name__)
        self.clean_memory          = CleanMemory(
            max_messages = self.max_messages,
            strategy     = "last",
            start_on     = "human"
        )

    def check_context(self, query: str, chat_history: List[str]) -> str:
        """
        Determine whether the current query requires additional context from chat history.

        Args:
            query (str): The user question to evaluate.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            str: Model response indicating whether context is needed (e.g., "Yes" or "No").
        """
        partial = self.check_context_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def contextualize_question(self, query: str, chat_history: List[str]) -> List[str]:
        """
        Rephrase the query to include relevant context from the chat history.

        Args:
            query (str): The original user question.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            List[str]: Contextualized query messages for downstream processing.
        """
        partial = self.contextualizer_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def process_and_reformulate_memory(self, input: Dict[str, str]) -> None:
        """
        Check if the input should use memory context and reformulate it if needed.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        response = self.check_context(
            query=input["input"],
            chat_history=self.memory.messages[-1]
        )
        self.logger.info(f"Context check result: {response}")

        if response.lower() in ("yes", "sim"):
            reformulated = self.contextualize_question(
                query=input["input"],
                chat_history=self.memory.messages[-1]
            )
            input["input"] = reformulated
            self.logger.info(f"Contextualized message: {input['input']}")

    def update_memory(self, input: Dict[str, str]) -> None:
        """
        Add the latest human message to memory and enforce memory size limits.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        self.memory.add_messages([HumanMessage(content=input["input"])])
        self.clean_memory.trim_messages(self.memory)

    def retrieve_document_as_a_list(self, input: Dict[str, str]) -> List:
        """
        Retrieve relevant documents for the given input query using the retriever.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the query string.

        Returns:
            List: Retrieved documents as a list.
        """
        query_text = input["input"]
        return self.retriever.invoke(query_text)

    def retrieved_documents(self) -> Runnable:
        """
        Build a runnable pipeline to fetch documents, optionally using chat history branching.

        Returns:
            Runnable: A configured retrieval pipeline.
        """
        return RunnableBranch(
            (
                lambda x: not x.get("chat_history", False),
                RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
            ),
            self.contextualizer_prompt | self.llm | StrOutputParser() |
            RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
        ).with_config(run_name="chat_retriever_chain")

    def build_conversational_chain(self) -> Runnable:
        """
        Create the full conversational RAG chain combining retrieval and question-answering.

        Returns:
            Runnable: A retrieval-augmented generation chain ready for invocation.
        """
        qa_chain = create_stuff_documents_chain(
            self.llm,
            self.system_message,
            document_variable_name="context"
        )
        return create_retrieval_chain(
            self.retrieved_documents(),
            qa_chain
        )

    def run(self, query: str) -> str:
        """
        Process a user query end-to-end: optional memory handling, retrieval, and response generation.

        Args:
            query (str): The user question to process.

        Returns:
            str: The generated response from the language model.
        """
        if self.include_memory and len(self.memory.messages) > 1:
            self.process_and_reformulate_memory({"input": query})

        response = self.build_conversational_chain().invoke({"input": query})
        self.logger.info(f"Input message: {query}")
        self.logger.info(f"Kb recovered: {self.retrieve_document_as_a_list({'input': query})}")
        self.logger.info(f"Response model: {response}")
        return response


In [9]:
bimo = Bimo(
    llm                   = llm, 
    system_message        = system_prompt, 
    check_context_prompt  = check_context_prompt, 
    contextualizer_prompt = contextualize_prompt, 
    memory                = InMemoryChatMessageHistory(), 
    retriever             = retriever, 
    include_memory        = True, 
    max_messages          = 5
)

### Interagindo com o modelo

In [10]:
%%time

"""
A saída da variável `response` se trata de um dicionário. 
A partir dela eu consigo ter os logs da mensagem enviada ao modelo, 
das bases de conhecimento utilizadas para suprir a resposta e também
da resposta do modelo. 

Tendo em vista que quero apenas verificar a resposta do modelo, 
visando garangir a experiência do usuário, irei retornar na 
saída do modelo a sua resposta e outros logs relacionados
"""


response = bimo.run(query="Olá, sobre o que pode falar comigo ?")
print(response)

{'input': 'Olá, sobre o que pode falar comigo ?', 'context': [Document(id='0a473b2b-1b9a-42d0-8be3-058a1d0a8155', metadata={'producer': 'iText® 5.3.5 ©2000-2012 1T3XT BVBA (SPRINGER SBM; licensed version)', 'creator': 'Springer', 'creationdate': '2023-12-19T21:26:00+05:30', 'keywords': '', 'moddate': '2023-12-19T19:15:56+01:00', 'subject': 'npj Digital Medicine, doi:10.1038/s41746-023-00979-5', 'author': 'Han Li', 'title': 'Systematic review and meta-analysis of AI-based conversational agents for promoting mental health and well-being', 'source': '../data/Review of AI and Mental Health.pdf', 'total_pages': 14, 'page': 8, 'page_label': '9'}, page_content='gical well-being did not\nexhibit signi ﬁcant variations associated with participants ’ age\n(F(2,12) = 1.444, p = 0.274), gender (F(1, 12) = 0.462, p = 0.51),\nhealth status (F(2, 12)= 1.624, p = 0.238), the response generation\napproach (F(2, 12) = 1.253, p = 0.32), interaction mode (F(2,\n12) = 1.338, p = 0.299) and delivery platfor

#### Interação 1°

In [10]:
%%time

response = bimo.run(query="Olá! Sobre o que posso falar contigo ?.")["answer"]

Markdown(response)

CPU times: total: 1.38 s
Wall time: 1.65 s


Olá! É um prazer ajudá-lo em seus estudos. Eu sou o Bimo, um assistente virtual treinado para auxiliar em estudos de artigos científicos e livros acadêmicos. Meu foco é em inteligência artificial, saúde mental e IA aplicada à saúde mental.

Você perguntou sobre o que podemos falar. Bem, podemos discutir sobre os artigos científicos que você está lendo, ou podemos explorar tópicos específicos dentro da área de inteligência artificial e saúde mental. Por exemplo, podemos falar sobre como os modelos generativos podem ser empregados na promoção da saúde mental, ou como a IA pode ser utilizada para melhorar a saúde mental.

Poderia me especificar sobre o que você gostaria de saber?

#### Interação 2°

In [11]:
%%time

response = bimo.run(query="Como as LLM's podem ser utilizadas no emprego da construção de chatbots?")["answer"]

Markdown(response)

CPU times: total: 1.3 s
Wall time: 1.55 s


Olá!

Entendi que você gostaria de saber como as LLMs (Large Language Models) podem ser utilizadas no emprego da construção de chatbots.

De acordo com o contexto fornecido, é possível criar chatbots que incluem funcionalidades de texto e voz, o que pode ser benéfico para indivíduos com deficiências cognitivas, linguísticas, de literacia ou motoras. Além disso, estudos sugerem que a eficácia do modalidade de chatbot pode depender do contexto e dos resultados desejados.

No contexto da saúde mental, chatbots podem ser utilizados para promover a saúde mental e bem-estar. Por exemplo, um estudo encontrou que chatbots baseados em texto foram mais eficazes em promover o consumo de frutas e legumes.

Portanto, é importante projetar chatbots que sejam personalizados e adaptáveis às necessidades específicas dos usuários.

Você gostaria de saber mais sobre como os modelos de linguagem podem ser utilizados para criar chatbots mais eficazes na promoção da saúde mental?

#### Interação 3°

In [12]:
%%time

response = bimo.run(query="Por que tais modelos podem ser utilizados para a construção de chatbots, e não outros ?")["answer"]

Markdown(response)

CPU times: total: 1.27 s
Wall time: 1.82 s


Olá! Eu sou o Bimo, seu assistente virtual para auxílio de estudos de artigos científicos e livros acadêmicos.

Entendi sua pergunta: "Por que tais modelos podem ser utilizados para a construção de chatbots, e não outros?"

Para responder à sua pergunta, vou utilizar o contexto fornecido. Os modelos de inteligência artificial (IA) mencionados no contexto são capazes de simular conversas humanas devido à sua capacidade de entender a intenção do usuário, analisar contextos e gerar respostas apropriadas. Isso os torna ideais para a construção de chatbots, pois podem fornecer respostas personalizadas e relevantes para os usuários.

Além disso, os modelos de IA podem ser treinados com grandes conjuntos de dados, o que lhes permite aprender e melhorar suas habilidades ao longo do tempo. Isso os torna mais eficazes em comparação com sistemas baseados em regras ou árvores de decisão, que dependem de regras pré-definidas para formular respostas.

Portanto, a combinação de habilidades de processamento de linguagem natural, aprendizado de máquina e capacidade de análise de contexto torna os modelos de IA ideais para a construção de chatbots.

Você gostaria de saber mais sobre como esses modelos de IA podem ser aplicados em saúde mental?

#### Interação 4° 

In [17]:
%%time

response = bimo.run(query="Mas imagino que não é todo e qualquer tipo de IA que pode ser utilizada, certo? Quais são as IA's ou modelos utilizados para a construção de chatbots?")["answer"]

Markdown(response)

CPU times: total: 1.42 s
Wall time: 1.75 s


Olá! Eu sou o Bimo, seu assistente virtual para auxílio de estudos de artigos científicos e livros acadêmicos.

Entendi sua pergunta: "Mas imagino que não é todo e qualquer tipo de IA que pode ser utilizada, certo? Quais são as IA's ou modelos utilizados para a construção de chatbots?"

Sim, você está correto. Nem todos os tipos de IA podem ser utilizados para construir chatbots. Os modelos de IA utilizados para construir chatbots são aqueles que possuem a capacidade de entender a intenção do usuário, analisar contextos e recuperar ou gerar respostas apropriadas com base na entrada do usuário e no contexto da conversa.

De acordo com o artigo de Abd-Alrazaq et al. (2019), os chatbots em saúde mental utilizam modelos de IA como NLP (Processamento de Linguagem Natural) e machine learning (aprendizado de máquina) para simular conversas humanas.

Além disso, os estudos incluídos na revisão de Koutsouleris et al. (2022) também destacam a importância da utilização de modelos de IA como NLP e machine learning para o desenvolvimento de chatbots.

Portanto, para construir chatbots, são necessários modelos de IA que possam entender a intenção do usuário e analisar contextos para fornecer respostas.

Você gostaria de saber mais sobre como esses modelos de IA são utilizados em saúde mental?

#### Interação 5° 

In [ ]:
%%time

response = bimo.run(query="Tais artigos mencionam o nome desses modelos de NLP empregados?")["answer"]

Markdown(response)